# Supplementary Tables — Element Statistics

This notebook assembles all supplementary tables from the descriptive element statistics
analysis (`000_element_statistics.ipynb`) into formatted TSV and XLSX files.

**Source data**: Intermediate CSV tables exported by `000_element_statistics.ipynb` to:
`paper/workflow/notebooks/figures/output/element_statistics_tables/`

**Output**: Combined XLSX workbook and individual TSV files written to:
`paper/workflow/notebooks/figures/output/supplementary_tables/`

## Table inventory

| Table | Description |
|-------|-------------|
| S1 | Transcript length descriptive statistics (PC vs lncRNA) |
| S2 | Transposable element (TE) counts by class |
| S3 | Top-20 TE families — mean hits per transcript |
| S4 | TE coverage statistics |
| S5 | TE insertion age (young vs ancient) |
| S6 | Low-complexity / tandem repeat (LCTR) element statistics |
| S7 | Non-B DNA (NBD) motif counts |
| S8 | NBD coverage — total and per-motif |
| S9 | Consolidated element enrichment (log₂ lncRNA/PC) |
| S10 | Statistical tests (Mann–Whitney U, BH-corrected) |

## 1. Imports and Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET = "gencode.v47.common.cdhit.cv"
BASE = Path("/mnt/cbib/LNClassifier/paper/results") / DATASET
SRC  = BASE / "supplementary_tables/element_statistics_tables"
OUT  = BASE / "supplementary_tables"
OUT.mkdir(parents=True, exist_ok=True)

# ── Table registry ───────────────────────────────────────────────────────────
TABLE_DESCRIPTIONS: dict[str, str] = {}

def load(name: str) -> pd.DataFrame:
    """Load an intermediate CSV from the element statistics export directory."""
    path = SRC / name
    if not path.exists():
        raise FileNotFoundError(
            f"{path}\n→ Run 000_element_statistics.ipynb first to generate this file."
        )
    return pd.read_csv(path)

print(f"Source : {SRC}")
print(f"Output : {OUT}")
print(f"Source files found: {len(list(SRC.glob('*.csv')))}")

## 2. Table S1 — Transcript Length Statistics

Descriptive statistics of transcript length for protein-coding and lncRNA transcripts
(GENCODE v49). Includes count, mean, standard deviation, quartiles, and median absolute
deviation (MAD). A Mann–Whitney U test comparing the two distributions is reported in
the analysis notebook.

In [ ]:
# ── S1: Transcript length statistics ──────────────────────────────────────────
s1 = load("S1_transcript_length_stats.csv")
s1 = s1.rename(columns={"Unnamed: 0": "Statistic"})

# Round numeric values
num_cols = [c for c in s1.columns if c != "Statistic"]
s1[num_cols] = s1[num_cols].apply(pd.to_numeric, errors="coerce").round(1)

TABLE_DESCRIPTIONS["S1"] = (
    "Transcript length descriptive statistics for protein-coding (PC) and lncRNA "
    "transcripts in GENCODE v49. Columns: count, mean, std, min, 25th/50th/75th "
    "percentiles, max, and median absolute deviation (MAD)."
)

s1.to_csv(OUT / "Table_S1_transcript_length.tsv", sep="\t", index=False)
display(s1)
print(f"\n✓ Table_S1_transcript_length.tsv written")

## 3. Table S2 — TE Element Counts by Class

Mean count, median count, and percentage of transcripts with ≥1 hit for each
transposable element class (DNA, LINE, LTR, SINE, ERV1, ERVK, ERVL, ERVL-MaLR,
RC, Retroposon, SRP RNA, PLE), stratified by coding class.

In [ ]:
# ── S2: TE counts by class ────────────────────────────────────────────────────
s2 = load("S2_te_counts_by_class.csv")

# Rename for clarity
s2 = s2.rename(columns={
    "coding_class": "Transcript class",
    "pct_present": "% transcripts with ≥1 hit",
})

# Round
for c in ["mean", "median", "% transcripts with ≥1 hit"]:
    if c in s2.columns:
        s2[c] = s2[c].round(4)

TABLE_DESCRIPTIONS["S2"] = (
    "Transposable element counts per TE class (DNA, LINE, LTR, SINE, ERV sub-families, "
    "RC, Retroposon, SRP RNA, PLE) for protein-coding and lncRNA transcripts. "
    "Columns: TE class, transcript class, mean count, median count, % of transcripts "
    "with at least one hit."
)

s2.to_csv(OUT / "Table_S2_te_counts_by_class.tsv", sep="\t", index=False)
display(s2)
print(f"\n✓ Table_S2_te_counts_by_class.tsv written")

## 4. Table S3 — Top-20 TE Families

Mean number of RepeatMasker hits per transcript for the 20 most abundant TE families,
reported separately for protein-coding and lncRNA transcripts. Families are ranked by
combined mean across both classes.

In [ ]:
# ── S3: Top-20 TE families (pivot: one row per family) ────────────────────────
s3_pivot = load("S3_te_top20_families_pivot.csv")
s3_pivot = s3_pivot.rename(columns={
    "rm_family": "TE family",
    "coding": "Mean hits/tx (PC)",
    "lncRNA": "Mean hits/tx (lncRNA)",
})

# Add log2 ratio
eps = 1e-6
s3_pivot["log2(lncRNA/PC)"] = np.log2(
    (s3_pivot["Mean hits/tx (lncRNA)"] + eps) /
    (s3_pivot["Mean hits/tx (PC)"] + eps)
)

for c in s3_pivot.columns[1:]:
    s3_pivot[c] = pd.to_numeric(s3_pivot[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S3"] = (
    "Top-20 transposable element families ranked by combined mean RepeatMasker hits "
    "per transcript. Columns: TE family, mean hits per transcript for protein-coding "
    "and lncRNA classes, and log₂ enrichment ratio (lncRNA/PC)."
)

s3_pivot.to_csv(OUT / "Table_S3_te_top20_families.tsv", sep="\t", index=False)
display(s3_pivot)
print(f"\n✓ Table_S3_te_top20_families.tsv written")

## 5. Table S4 — TE Coverage Statistics

Summary of TE sequence coverage per transcript: total hit length (nucleotides and
percentage of transcript), TE density (hits per kilobase), mean hit length, and mean
reference coverage of hits, for protein-coding and lncRNA transcripts.

In [ ]:
# ── S4: TE coverage statistics ────────────────────────────────────────────────
s4 = load("S4_te_coverage_stats.csv")

for c in ["Mean", "Median", "SD", "% > 0"]:
    if c in s4.columns:
        s4[c] = pd.to_numeric(s4[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S4"] = (
    "Transposable element coverage statistics for protein-coding and lncRNA "
    "transcripts. Metrics: total hit length (nt and % of transcript), TE density "
    "(hits/kb), mean hit length (nt), and mean reference coverage per hit. "
    "Columns: metric, class, mean, median, SD, and % of transcripts with value > 0."
)

s4.to_csv(OUT / "Table_S4_te_coverage.tsv", sep="\t", index=False)
display(s4)
print(f"\n✓ Table_S4_te_coverage.tsv written")

## 6. Table S5 — TE Insertion Age

Comparison of young versus ancient transposable element insertions between
protein-coding and lncRNA transcripts, based on divergence thresholds applied to
RepeatMasker alignments.

In [ ]:
# ── S5: TE insertion age ──────────────────────────────────────────────────────
s5 = load("S5_te_age_stats.csv")

# Clean up multi-index columns from CSV serialization
if "Unnamed: 0" in s5.columns:
    s5 = s5.rename(columns={"Unnamed: 0": "Age"})
if "Unnamed: 1" in s5.columns:
    s5 = s5.rename(columns={"Unnamed: 1": "Class"})

for c in s5.columns:
    if c not in ("Age", "Class"):
        s5[c] = pd.to_numeric(s5[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S5"] = (
    "Young versus ancient transposable element insertion counts for protein-coding "
    "and lncRNA transcripts. Columns: age category, transcript class, mean count, "
    "median count, mean count per kilobase, and % of transcripts with at least one hit."
)

s5.to_csv(OUT / "Table_S5_te_age.tsv", sep="\t", index=False)
display(s5)
print(f"\n✓ Table_S5_te_age.tsv written")

## 7. Table S6 — LCTR Element Statistics

Statistics for low-complexity and tandem repeat (LCTR) elements, including total LCTR,
low-complexity, simple repeat, and satellite sub-types. Reported per transcript class.

In [ ]:
# ── S6: LCTR statistics ───────────────────────────────────────────────────────
s6 = load("S6_lctr_stats.csv")

if "Unnamed: 0" in s6.columns:
    s6 = s6.rename(columns={"Unnamed: 0": "LCTR type"})
if "Unnamed: 1" in s6.columns:
    s6 = s6.rename(columns={"Unnamed: 1": "Class"})

for c in s6.columns:
    if c not in ("LCTR type", "Class"):
        s6[c] = pd.to_numeric(s6[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S6"] = (
    "Low-complexity / tandem repeat (LCTR) element statistics for protein-coding and "
    "lncRNA transcripts. Sub-types: total LCTR, low complexity, simple repeat, satellite. "
    "Columns: LCTR type, class, mean count, median count, mean count per kilobase, "
    "and % of transcripts with at least one hit."
)

s6.to_csv(OUT / "Table_S6_lctr_stats.tsv", sep="\t", index=False)
display(s6)
print(f"\n✓ Table_S6_lctr_stats.tsv written")

## 8. Table S7 — Non-B DNA Motif Counts

Per-motif statistics for non-B DNA (NBD) structural motifs: APR, DR, G4, IR, MR, STR,
TRI, and Z-DNA. Includes mean/median counts, density (count per kb), mean coverage (%),
and fraction of transcripts bearing each motif type.

In [ ]:
# ── S7: NBD motif counts ──────────────────────────────────────────────────────
s7 = load("S7_nbd_counts.csv")

for c in s7.columns:
    if c not in ("Motif", "Class"):
        s7[c] = pd.to_numeric(s7[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S7"] = (
    "Non-B DNA motif statistics per motif type (APR, DR, G4, IR, MR, STR, TRI, Z-DNA) "
    "for protein-coding and lncRNA transcripts. Columns: motif, class, mean count, "
    "median count, mean count per kilobase, mean coverage (%), and % of transcripts "
    "with at least one hit."
)

s7.to_csv(OUT / "Table_S7_nbd_counts.tsv", sep="\t", index=False)
display(s7)
print(f"\n✓ Table_S7_nbd_counts.tsv written")

## 9. Table S8 — Non-B DNA Coverage

**S8a**: Total non-B DNA coverage summary (aggregate across all motif types).
**S8b**: Per-motif coverage statistics including mean count, coverage (%), SD, MAD,
median, IQR, and % of transcripts with non-zero coverage.

In [ ]:
# ── S8: NBD coverage ──────────────────────────────────────────────────────────
s8a = load("S8a_nbd_total_coverage.csv")
s8b = load("S8b_nbd_per_motif_coverage.csv")

# Round numeric columns
for df in [s8a, s8b]:
    for c in df.columns:
        if c not in ("Class", "Motif"):
            df[c] = pd.to_numeric(df[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S8a"] = (
    "Total non-B DNA coverage summary (all motif types combined) for protein-coding "
    "and lncRNA transcripts. Includes mean/SD/MAD/median/IQR for total count and "
    "coverage (%), mean number of distinct motif types present, and % of transcripts "
    "with any non-B DNA motif."
)
TABLE_DESCRIPTIONS["S8b"] = (
    "Per-motif non-B DNA coverage statistics. For each motif type (APR, DR, G4, IR, "
    "MR, STR, TRI, Z-DNA) and transcript class: mean count, median count, mean/SD/"
    "MAD/median/IQR of coverage (%), and % of transcripts with coverage > 0."
)

s8a.to_csv(OUT / "Table_S8a_nbd_total_coverage.tsv", sep="\t", index=False)
s8b.to_csv(OUT / "Table_S8b_nbd_per_motif_coverage.tsv", sep="\t", index=False)

print("── S8a: Total NBD coverage ──")
display(s8a)
print("\n── S8b: Per-motif NBD coverage ──")
display(s8b)
print(f"\n✓ Table_S8a/S8b written")

## 10. Table S9 — Consolidated Element Enrichment

Log₂ enrichment ratios (lncRNA / protein-coding) for all element types: transcript
length, TE classes, LCTR sub-types, and NBD motif types. Positive values indicate
enrichment in lncRNA transcripts.

In [ ]:
# ── S9: Consolidated element enrichment ───────────────────────────────────────
s9 = load("S9_consolidated_element_stats.csv")

# Select and rename key columns
s9_export = s9[["Group", "Feature", "PC mean", "lncRNA mean", "log2(lncRNA/PC)"]].copy()
s9_export = s9_export.rename(columns={
    "PC mean": "Mean (PC)",
    "lncRNA mean": "Mean (lncRNA)",
})

for c in ["Mean (PC)", "Mean (lncRNA)", "log2(lncRNA/PC)"]:
    s9_export[c] = pd.to_numeric(s9_export[c], errors="coerce").round(4)

TABLE_DESCRIPTIONS["S9"] = (
    "Consolidated element statistics across all feature groups (Length, TE, LCTR, NBD). "
    "For each feature: mean value for protein-coding and lncRNA transcripts, and "
    "log₂ enrichment ratio (lncRNA / PC). Positive log₂ values indicate higher "
    "abundance in lncRNA transcripts."
)

s9_export.to_csv(OUT / "Table_S9_element_enrichment.tsv", sep="\t", index=False)
display(s9_export)
print(f"\n✓ Table_S9_element_enrichment.tsv written")

## 11. Table S10 — Statistical Tests

Mann–Whitney U tests comparing protein-coding and lncRNA transcripts for each element
type, with Benjamini–Hochberg (BH) correction for multiple testing. Includes the U
statistic, raw p-value, adjusted p-value, significance flag, and log₂ enrichment ratio.

In [ ]:
# ── S10: Statistical tests ────────────────────────────────────────────────────
s10 = load("S10_statistical_tests.csv")

# Format columns
s10 = s10.rename(columns={
    "p_raw": "p (raw)",
    "p_adj_BH": "p (BH-adjusted)",
    "significant": "Significant (α=0.05)",
})

# Select informative columns
s10_export = s10[["Group", "Feature", "U", "p (raw)", "p (BH-adjusted)",
                   "Significant (α=0.05)", "log2(lncRNA/PC)"]].copy()

for c in ["U", "p (raw)", "p (BH-adjusted)", "log2(lncRNA/PC)"]:
    s10_export[c] = pd.to_numeric(s10_export[c], errors="coerce")

# Sort by adjusted p-value
s10_export = s10_export.sort_values("p (BH-adjusted)")

TABLE_DESCRIPTIONS["S10"] = (
    "Mann–Whitney U tests comparing protein-coding vs lncRNA transcripts for each "
    "element type (TE classes, LCTR sub-types, NBD motifs). Columns: feature group, "
    "feature name, U statistic, raw p-value, BH-adjusted p-value, significance at "
    "α = 0.05, and log₂ enrichment ratio. Sorted by adjusted p-value."
)

s10_export.to_csv(OUT / "Table_S10_statistical_tests.tsv", sep="\t", index=False)
display(s10_export)
print(f"\n✓ Table_S10_statistical_tests.tsv written")

## 12. Combined XLSX Workbook

All tables are collected into a single Excel workbook with one sheet per table.
A `_README` sheet provides descriptions of each table.

In [ ]:
# ── Combined XLSX workbook ────────────────────────────────────────────────────
xlsx_path = OUT / "Supplementary_Tables_Element_Statistics.xlsx"

# Collect all tables with their sheet names
sheets: dict[str, pd.DataFrame] = {
    "S1_Length": s1,
    "S2_TE_Counts": s2,
    "S3_TE_Families": s3_pivot,
    "S4_TE_Coverage": s4,
    "S5_TE_Age": s5,
    "S6_LCTR": s6,
    "S7_NBD_Counts": s7,
    "S8a_NBD_Total_Cov": s8a,
    "S8b_NBD_Motif_Cov": s8b,
    "S9_Enrichment": s9_export,
    "S10_Statistics": s10_export,
}

# Build README sheet
readme_rows = []
for key, desc in TABLE_DESCRIPTIONS.items():
    sheet_name = next((k for k in sheets if k.startswith(key)), key)
    readme_rows.append({"Sheet": sheet_name, "Table": key, "Description": desc})
readme_df = pd.DataFrame(readme_rows)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    readme_df.to_excel(writer, sheet_name="_README", index=False)
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✓ Combined workbook written: {xlsx_path}")
print(f"  Sheets: {', '.join(['_README'] + list(sheets.keys()))}")
print(f"  File size: {xlsx_path.stat().st_size / 1024:.1f} KB")

## 13. Summary

In [ ]:
# ── Summary of all exported tables ────────────────────────────────────────────
print("=" * 80)
print("SUPPLEMENTARY TABLE SUMMARY — Element Statistics")
print("=" * 80)

tsv_files = sorted(OUT.glob("Table_S*.tsv"))
print(f"\nTotal TSV files exported: {len(tsv_files)}")
print(f"XLSX workbook: Supplementary_Tables_Element_Statistics.xlsx")
print(f"Output directory: {OUT}\n")

for tsv in tsv_files:
    df = pd.read_csv(tsv, sep="\t", nrows=0)
    n_rows = sum(1 for _ in open(tsv)) - 1
    print(f"  {tsv.name:<50s}  {n_rows:>4d} rows × {len(df.columns):>3d} cols")

print("\n" + "-" * 80)
print("TABLE DESCRIPTIONS")
print("-" * 80)
for key, desc in TABLE_DESCRIPTIONS.items():
    print(f"\n{key}: {desc}")